### 閃亜鉛鉱構造とウルツ鉱構造の安定性の分類

**データ取得からデータ解析**


In [ ]:
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')


%matplotlib inline
pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 60)


In [ ]:
# データ取得
def get_data():
    df = pd.read_csv("../data/ZB_WZ_dE_rawdescriptor.csv")
    descriptor_names = ['IP_A', 'EA_A', 'EN_A', 'Highest_occ_A',
                         'Lowest_unocc_A', 'rs_A', 'rp_A', 'rd_A', 'IP_B', 'EA_B', 'EN_B',
                         'Highest_occ_B', 'Lowest_unocc_B', 'rs_B', 'rp_B', 'rd_B']
    target_name = "dE"
    return df, descriptor_names, target_name

g_df, g_descriptor_names, g_target_name = get_data()

次の関数で分類問題用に
```python
y = yraw >0 
```
として目的変数を２クラスに変換しています。

In [ ]:
def classify_df(df, descriptor_names, target_name):
    """データを分類する。

    Args:
        df (pd.DataFrame): データ
        descriptor_names ([str]): 説明変数名リスト
        target_name ([str])): 目的変数名リスト

    Returns:
        LogisticRegressionCV: LogisticRegressionCVインスタンス .
        np.ndarray: X.
        np.ndarray: y.
        np.ndarray: values of cls.predict(X)
        np.ndarray: values of cls.predict_proba(X)
    """
    Xraw = df.loc[:, descriptor_names].values
    yraw = df.loc[:, target_name].values
    y = yraw > 0

    # データプリプロセス
    scaler = StandardScaler()
    scaler.fit(Xraw)
    X = scaler.transform(Xraw)

    # データ解析
    kf = KFold(5, shuffle=True)
    cls = LogisticRegressionCV(cv=kf)
    cls.fit(X, y)
    score = cls.score(X, y)
    print("score=", score)
    yp = cls.predict(X)
    yp_proba = cls.predict_proba(X)
    print(classification_report(y, yp))
    index = []
    columns= []
    for s in cls.classes_:
        index.append("actual({})".format(s))
        columns.append("predict({})".format(s))
    cmdf = pd.DataFrame(confusion_matrix(y, yp), index=index, columns=columns)
    display(cmdf)
    
    return cls, X, y, yp, yp_proba

g_cls, g_X, g_y, g_yp, g_yp_proba =  classify_df(g_df, g_descriptor_names, g_target_name)

**可視化**

In [ ]:
g_df

In [ ]:
def plot_X(X):
    """説明変数の図示。

    Args:
        X (np.ndarray): 説明変数
    """
    fig, ax = plt.subplots()
    ax.plot(X)
    ax.set_xlabel("index")
    ax.set_ylabel("X")
    
plot_X(g_X)

from collections import Counter
print("Counter", Counter(g_y))

In [ ]:
g_Copt = g_cls.C_[0]
print("Copt=", g_Copt)

二値の場合はTrueに対して値が入っています。

CVの可視化を行います。

In [ ]:
def plot_CV_scores(cls):
    """cls.scoreの表示。

    Args:
        cls (LogisticRegressionCV): LogisticRegressionCVインスタンス.

    """
    scores_mean = np.mean(cls.scores_[True], axis=0)
    scores_std = np.std(cls.scores_[True], axis=0)
    ic = np.argmax(scores_mean)
    print("index=", ic, "score=", scores_mean[ic])

    fig, ax = plt.subplots()
    ax.errorbar(np.log10(cls.Cs_), scores_mean, yerr=scores_std, capsize=5)
    ax.set_xlabel("log10(C)")
    ax.set_ylabel("score")
    
plot_CV_scores(g_cls)

### 付録
boxplotでも見ておきます。

In [ ]:
def plot_CV_scores_as_boxplot(cls, save_fig: bool=False):
    """cls.scoreの表示をboxplotで行う。

    Args:
        cls (LogisticRegressionCV): LogisticRegressionCVインスタンス.

    """    
    labels = []
    for x in np.log10(cls.Cs_):
        labels.append("{:.3f}".format(x))
    df_score = pd.DataFrame(cls.scores_[True], columns=labels)
    fig, ax = plt.subplots()
    df_score.boxplot(rot=90, ax=ax)
    ax.set_xlabel("log10(C)")
    if save_fig:
        fig.savefig("image_executed/ZB_WZ_cls_boxplot.png")
    
plot_CV_scores_as_boxplot(g_cls)

### CV(test)の表示

LogisticRegressionCVは観測データ全てを用いた評価値なので、
CV(test)だけを表示する。


In [ ]:
def calc_CV_score(X,y, Copt):
    """CVを行い、score、y, ypを出力する。

    Args:
        X (np.ndarray)): 説明変数
        y (np.ndarray): 目的変数
        Copt (float): C of logistic regression.
        
    Returns:
        [float]: a list of KFold scores.
        [np.array]: a list of KFold y_test.
        [np.array]: a list of KFold predicted y_test.

    """
    ytest_list = []
    ytestp_list = []
    score_list = []
    kf = KFold(5, shuffle=True)
    for train, test in kf.split(X):
        Xtrain, ytrain = X[train], y[train]
        Xtest, ytest = X[test], y[test]
        cls = LogisticRegression(C=Copt)
        cls.fit(Xtrain, ytrain)
        ytestp = cls.predict(Xtest)
        ytest_list.extend(ytest)
        ytestp_list.extend(ytestp)
        score = cls.score(Xtest, ytest)
        score_list.append(score)
    return score_list, ytest_list, ytestp_list

g_score_list, g_ytest_list, g_ytestp_list = calc_CV_score(g_X, g_y, g_Copt)

In [ ]:
def show_scores(score_list, ytest_list, ytestp_list, classes):
    """score, y, ypの図示。

    Args:
        score_list ([float]]): a list of KFold scores.
        ytest_list ([np.array]): a list of y_test
        ytestp_list ([np.array]): a list of predicted y_test
        classes ([int]): classification classes.
    """
    import os
    print("score = {}({})".format(np.mean(score_list), np.std(score_list)))
    print(classification_report(ytest_list, ytestp_list))
    os.makedirs("image_executed", exist_ok=True)
    with open("image_executed/ZB_WZ_cls_report.txt", "w") as f:
        f.write((classification_report(ytest_list, ytestp_list)))
    index = []
    columns= []
    for s in classes:
        index.append("actual({})".format(s))
        columns.append("predict({})".format(s))
    df_cm = pd.DataFrame(confusion_matrix(ytest_list, ytestp_list), index=index,
                         columns=columns)
    display(df_cm)
    
show_scores(g_score_list, g_ytest_list, g_ytestp_list, g_cls.classes_ )

**参考文献**

1. Luca M. Ghiringhelli,
Jan Vybiral,
Sergey V. Levchenko,
Claudia Draxl,
and Matthias Scheffler,
"Big Data of Materials Science: Critical Role of the Descriptor",
Phys. Rev. Lett. 114, 105503 (2015)


## 問題

../data_calculated/ZB_WZ_dE_3var.csvを使って同じことを行え。